# 2.0 Clean and Join

**Class 2.** Raw data is almost never ready to analyze. In this notebook we turn the six raw files from notebook 1.1 into **one analysis-ready table**: one row per person, one column per variable. Data cleaning like this usually takes more of a research project's time than the analysis itself.

### By the end of this notebook you'll be able to

1. Create a **flagging variable** that records which group each person is in
2. **Stack** tables with `bind_rows()` and **join** them with `left_join()`, and know when to use which
3. Reshape data from **long** to **wide** with `pivot_wider()`
4. Clean text with **pattern matching** (`str_detect()`, `case_when()`)
5. Keep each person's **most recent** measurement

| | |
|---|---|
| ⏱️ **Time** | 60–75 minutes |
| 📂 **Uses** | `person_study.csv`, `person_control.csv`, `survey_study.csv`, `survey_control.csv`, `measurement_study.csv`, `measurement_control.csv` |
| 💾 **Saves** | `final_analysis.csv`, `measurements_wide.csv` |

**Before you start:**

- Run **0.0 setup**
- The six files from **1.1** in your bucket (or files from the earlier notebooks, which use the same names)

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## The plan

| Step | What we do | Main functions |
|---|---|---|
| 1 | Load the six raw files | `load_data()` |
| 2 | Create a **flagging variable**: which group is each person in? | `mutate()` |
| 3 | **Bring the groups together** (stack them) | `bind_rows()` |
| 4 | Survey: tidy the answers, one column per question | `case_when()`, `pivot_wider()` |
| 5 | Measurements: group the names, keep the latest value, one column each | `case_when()`, `slice_max()`, `pivot_wider()` |
| 6 | **Join** everything and add age and age group | `left_join()`, `cut()` |
| 7 | Save `final_analysis.csv` | `save_data()` |

## Key concepts

**Tidy data.** Analysis functions in R expect **tidy** data: each **row** is one observation (here, one person), each **column** is one variable, and each cell holds one value. Our raw files aren't tidy yet: the survey and measurement files have **many rows per person**.

**Long vs wide.** The same information can be laid out two ways:

**Long:** one row per person *per question*

| person_id | question | answer |
|---|---|---|
| 101 | ptsd_doctor | Yes |
| 101 | ptsd_treatment | No |
| 102 | ptsd_doctor | No |
| 102 | ptsd_treatment | No |

**Wide:** one row per person, one column per question

| person_id | ptsd_doctor | ptsd_treatment |
|---|---|---|
| 101 | Yes | No |
| 102 | No | No |

Databases store data **long**; most analyses need it **wide**.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

## Step 1: Load the raw data

Notebook 1.1 saved each group to its own file: `_study` (has diabetes) and `_control` (no diabetes). `load_data()` (our helper) reads a CSV file from your bucket's `data/` folder.

In [ ]:
person_study        <- load_data("person_study.csv")
person_control      <- load_data("person_control.csv")
survey_study        <- load_data("survey_study.csv")
survey_control      <- load_data("survey_control.csv")
measurement_study   <- load_data("measurement_study.csv")
measurement_control <- load_data("measurement_control.csv")

glimpse(person_study)

## Step 2: Create a flagging variable

When we combine the two groups into one table, we'd lose track of **who came from which group**, unless we record it first. A **flagging variable** (or *indicator*) is a column that marks each row as belonging to a group. Ours is called `diabetes`:

- `"Yes"` for everyone in the **study** group
- `"No"` for everyone in the **control** group

`mutate()` (from **dplyr**) adds a new column, or changes an existing one. Giving it a single value fills that value into every row.

In [ ]:
person_study   <- person_study   %>% mutate(diabetes = "Yes")
person_control <- person_control %>% mutate(diabetes = "No")

head(person_study, 3)
head(person_control, 3)

**🔍 Read the output:** both tables now end with a `diabetes` column, `"Yes"` in one and `"No"` in the other. (`head(df, 3)` shows the first 3 rows.)

> 💡 **Why only the person tables?** The survey and measurement tables are linked to a group through `person_id` when we join everything together in Step 6. One flag per person is enough.

Later, in the regression notebook, you'll see the same flag turned into numbers: 1 for *Yes*, 0 for *No*.

## Step 3: Bring the groups together

The study and control tables have **the same columns**, so we **stack** them: `bind_rows()` (from **dplyr**) puts the control rows underneath the study rows. There are two ways to combine tables, and choosing the right one matters:

| | `bind_rows()`: **stacking** | `left_join()`: **joining** |
|---|---|---|
| Adds | **rows**: more people, same columns | **columns**: more variables for the same people |
| Use when | the tables have the **same columns** | the tables share an **ID** (`person_id`) |
| Here | study + control groups (now) | demographics + survey + measurements (Step 6) |

At the same time, `transmute()` keeps only the columns we need, in the order we want, and turns `date_of_birth` from a date-*time* into a plain date with `as.Date()`.

In [ ]:
person <- bind_rows(person_study, person_control) %>%
  transmute(person_id, diabetes, date_of_birth = as.Date(date_of_birth), gender, race, ethnicity)

survey      <- bind_rows(survey_study, survey_control)
measurement <- bind_rows(measurement_study, measurement_control)

person %>% count(diabetes)

**🔍 Read the output:** `count(diabetes)` shows how many people are in each group. The two numbers should add up to the total number of rows in `person`.

**Check:** nobody should be in **both** groups. `intersect()` (base R) finds values that appear in both lists; `length()` counts them. The answer should be **0**. If it isn't, something went wrong when building the cohorts.

In [ ]:
length(intersect(person_study$person_id, person_control$person_id))

`$` pulls one column out of a data frame: `person_study$person_id` is the list of IDs in the study group.

### ✍️ Your turn

What would go wrong if we had used `left_join()` instead of `bind_rows()` to combine `person_study` and `person_control`?

<details>
<summary><b>Show answer</b></summary>

A join matches rows **by ID**. Nobody is in both groups, so no IDs match: `left_join(person_study, person_control, by = "person_id")` would keep only the study group, with empty (`NA`) copies of every control column. The control group would vanish.

**Stack** when the tables hold *different people* with the *same columns*; **join** when they hold *different information* about the *same people*.

</details>

In [ ]:
# Your code here (optional: try it and look at the result)

## Step 4: Survey answers

Right now `survey` is **long**: one row per person *per question*. We want **one row per person**, with a column for each question.

**4a. Short names and clean answers.** The questions are long sentences, and the answers come in several forms. We:

- give each question a **short name**: `ptsd_doctor` or `ptsd_treatment`
- reduce each answer to **"Yes"** or **"No"**, and drop everything else (*Skip*, *Prefer not to answer*…)

`case_when()` works like a series of *if… then…* rules, checked in order; the first rule that's `TRUE` wins, and `TRUE ~ …` at the end catches everything else. `str_detect()` checks whether text contains a **pattern**:

- `"prescribed|treatment"`: the `|` means **or**
- `"\\byes\\b"`: `\\b` marks a **word boundary**, so it matches the word *yes* but not, say, *eyes*
- `regex(…, ignore_case = TRUE)`: upper and lower case count as the same

In [ ]:
survey_clean <- survey %>%
  mutate(
    question = case_when(
      str_detect(question, regex("prescribed|treatment", ignore_case = TRUE)) ~ "ptsd_treatment",
      str_detect(question, regex("doctor|provider", ignore_case = TRUE))      ~ "ptsd_doctor",
      TRUE ~ NA_character_
    ),
    answer = case_when(
      str_detect(answer, regex("\\byes\\b", ignore_case = TRUE)) ~ "Yes",
      str_detect(answer, regex("\\bno\\b",  ignore_case = TRUE)) ~ "No",
      TRUE ~ NA_character_
    )
  ) %>%
  filter(!is.na(question), !is.na(answer)) %>%
  distinct(person_id, question, .keep_all = TRUE)   # one answer per person per question

survey_clean %>% count(question, answer)

**🔍 Read the output:** four rows: each question with *Yes* and *No*. If you see other values, the wording in your data differs; look at `survey %>% distinct(question, answer)` and adjust the patterns.

- `filter(!is.na(…))` keeps rows where the value is **not** missing (`!` means *not*)
- `distinct(person_id, question, .keep_all = TRUE)` keeps one answer per person per question, in case someone answered twice

**4b. Long to wide.** `pivot_wider()` (from **tidyr**) turns the values of `question` into **column names**, and fills them with the values of `answer`.

In [ ]:
survey_wide <- survey_clean %>%
  pivot_wider(names_from = question, values_from = answer)

head(survey_wide)

**🔍 Read the output:** now one row per person, with `ptsd_doctor` and `ptsd_treatment` columns. Someone who answered only one question has `NA` in the other.

## Step 5: Measurements

**5a. Group the measurement names.** There are many slightly different names for the same measurement (several kinds of systolic blood pressure, LDL measured in different ways). See for yourself:

In [ ]:
measurement %>% count(standard_concept_name, sort = TRUE)

We collapse them into **four groups** with `case_when()`. Order matters: the first matching rule wins. Anything that doesn't match becomes `"Other"`, so we can check nothing important slipped through.

In [ ]:
measurement_grouped <- measurement %>%
  mutate(measure_group = case_when(
    str_detect(standard_concept_name, regex("body mass index", ignore_case = TRUE))                     ~ "BMI",
    str_detect(standard_concept_name, regex("systolic", ignore_case = TRUE))                            ~ "Systolic_BP",
    str_detect(standard_concept_name, regex("a1c|glycated hemoglobin|glycohemoglobin", ignore_case = TRUE)) ~ "HbA1c",
    str_detect(standard_concept_name, regex("ldl", ignore_case = TRUE))                                 ~ "LDL_Cholesterol",
    TRUE ~ "Other"
  ))

measurement_grouped %>% count(measure_group)

**🔍 Read the output:** four groups, ideally with no (or very few) `Other`. If `Other` is large, look at which names ended up there: `measurement_grouped %>% filter(measure_group == "Other") %>% count(standard_concept_name)`.

> 💡 **Pro tip:** with many messy names, you can paste the list from the previous cell into an AI assistant and ask it to suggest groups. Only the **names** (they're dictionary entries, not participant data), never the data itself.

**5b. Keep the most recent value.** Most people have many measurements of each kind over the years. We keep the **latest** one per person per measurement:

- `group_by(person_id, measure_group)` makes every following step happen **within** each person-and-measurement group
- `slice_max(measurement_datetime, n = 1)` keeps the row with the latest date in each group; `with_ties = FALSE` keeps just one if two share a date
- `ungroup()` switches grouping off again (always do this when you're done)

In [ ]:
measurement_latest <- measurement_grouped %>%
  filter(measure_group != "Other") %>%
  group_by(person_id, measure_group) %>%
  slice_max(measurement_datetime, n = 1, with_ties = FALSE) %>%
  ungroup()

nrow(measurement_grouped)
nrow(measurement_latest)

**🔍 Read the output:** the number of rows drops a lot: from millions of measurements to at most four per person.

**5c. Long to wide:** one column per measurement.

In [ ]:
measurements_wide <- measurement_latest %>%
  pivot_wider(id_cols = person_id, names_from = measure_group, values_from = value_as_number)

head(measurements_wide)

We save this table too: the survival analysis (3.4) uses it.

In [ ]:
save_data(measurements_wide, "measurements_wide.csv")

### ✍️ Your turn

We kept the **most recent** value. Change the code to keep each person's **earliest** measurement instead. (Hint: there's a function called `slice_min()`.)

<details>
<summary><b>Show answer</b></summary>

```r
measurement_earliest <- measurement_grouped %>%
  filter(measure_group != "Other") %>%
  group_by(person_id, measure_group) %>%
  slice_min(measurement_datetime, n = 1, with_ties = FALSE) %>%
  ungroup()
```

Which one is right depends on the research question. For a study of risk factors, you'd usually want the value from **before** the outcome; we use the most recent one here to keep things simple.

</details>

In [ ]:
# Your code here

## Step 6: Join everything

Now we **join**: `left_join()` (from **dplyr**) keeps **every** person in `person` (with their `diabetes` flag) and adds the survey and measurement columns that match on `person_id`. People with no match get `NA`.

Then we:

- calculate **age** in whole years: `interval()` and `time_length()` (from **lubridate**) measure the time from date of birth to `today()`, and `floor()` rounds down
- make **age groups** with `cut()`, which sorts numbers into ranges: `breaks = c(25, 35, 45, 51)` with `right = FALSE` means 25–34, 35–44, and 45–50
- `select()` the columns we want, in a sensible order
- keep only people with **complete data** with `drop_na()`, so every analysis uses the same people

In [ ]:
final_analysis <- person %>%
  left_join(survey_wide,       by = "person_id") %>%
  left_join(measurements_wide, by = "person_id") %>%
  mutate(
    age = floor(time_length(interval(date_of_birth, today()), "years")),
    age_group = cut(age, breaks = c(25, 35, 45, 51), right = FALSE,
                    labels = c("25-34", "35-44", "45-50"))
  ) %>%
  select(person_id, diabetes, age, age_group, gender, race, ethnicity,
         ptsd_doctor, ptsd_treatment, BMI, Systolic_BP, LDL_Cholesterol, HbA1c) %>%
  drop_na()

glimpse(final_analysis)

**🔍 Read the output:** one row per person, 13 columns, no missing values. This is far fewer people than in `person`: most people didn't answer both PTSD questions, or don't have all four measurements.

> 🤔 **Think about it:** `drop_na()` removes everyone missing *any* variable. This is called a **complete-case analysis**. How might the people we keep differ from the people we drop? (For example: people who answer PTSD questions, or who have regular lab tests, may be different from people who don't.) This can **bias** results, and it's worth mentioning as a limitation.

### ✍️ Your turn

How many people did `drop_na()` remove? Compare `nrow(person)` with `nrow(final_analysis)`. Then use `count(diabetes)` to see whether one group lost more people than the other.

<details>
<summary><b>Show answer</b></summary>

```r
nrow(person)
nrow(final_analysis)

person %>% count(diabetes)
final_analysis %>% count(diabetes)
```

If one group loses a much bigger share, that's a sign the complete cases might not represent the whole group equally.

</details>

In [ ]:
# Your code here

## Step 7: Save

In [ ]:
save_data(final_analysis, "final_analysis.csv")

## 📦 Functions used in this notebook

| Function | Package | What it does |
|---|---|---|
| `mutate()` | dplyr | adds or changes columns |
| `bind_rows()` | dplyr | **stacks** tables with the same columns |
| `transmute()` | dplyr | like `mutate()`, but keeps only the columns you name |
| `intersect()` | base R | values found in both lists |
| `case_when()` | dplyr | *if… then…* rules, checked in order |
| `str_detect()`, `regex()` | stringr | does the text contain a pattern? |
| `filter()` | dplyr | keeps rows that meet a condition |
| `distinct()` | dplyr | removes duplicate rows |
| `pivot_wider()` | tidyr | long → wide |
| `group_by()`, `ungroup()` | dplyr | do the next steps within groups |
| `slice_max()`, `slice_min()` | dplyr | keep the row with the highest / lowest value |
| `left_join()` | dplyr | **joins** tables by a shared ID |
| `interval()`, `time_length()`, `today()` | lubridate | work with dates and durations |
| `cut()` | base R | sorts numbers into ranges |
| `drop_na()` | tidyr | removes rows with any missing value |

## ⚠️ Common errors

| Problem | Fix |
|---|---|
| *'person_study.csv' does not exist* | Run 1.1 first, or check `BUCKET_NAME` in `aou_setup.R` |
| `survey_clean` has no rows | The answer wording differs; check `survey %>% distinct(question, answer)` |
| A measurement column is missing from `final_analysis` | No measurement matched that group; check the `measure_group` counts |

## ✅ Key takeaways

- A **flagging variable** records group membership **before** you combine groups
- **Stack** (`bind_rows()`) to add people; **join** (`left_join()`) to add variables
- Analyses need **tidy, wide** data: one row per person
- Every cleaning choice (latest value, complete cases) is a **decision** that can affect your results: write it down

**Next:** **3.1 table_one**.